# 07 — Inférence en temps réel

Exécute les cellules dans l’ordre avec le noyau Poetry du projet.
Prérequis : `poetry install` et modèle approuvé publié depuis le notebook 05.
Explications détaillées : [README de l’API](../README_API.md).

## 1. Vérifier l’environnement

Exécute la cellule. Attendu : Python 3.11 ou 3.12 et les versions des paquets.
Si un paquet manque, lance `poetry install` puis sélectionne le noyau Poetry.

In [6]:
import sys
from importlib.metadata import PackageNotFoundError
from importlib.metadata import version as package_version

print(f"Python du notebook : {sys.executable}")
print(f"Version Python : {sys.version.split()[0]}")
assert (3, 11) <= sys.version_info[:2] < (3, 13), (
    "Ce projet attend Python 3.11 ou 3.12. Sélectionne le noyau Poetry du projet."
)

# Lire les versions ne démarre aucun serveur et ne charge aucun modèle.
for package_name in ["bnpl-credit-risk", "fastapi", "uvicorn", "httpx", "pydantic"]:
    try:
        installed_version = package_version(package_name)
    except PackageNotFoundError as exc:
        raise RuntimeError(
            f"{package_name} manque dans ce noyau. Exécute poetry install dans le terminal "
            "puis sélectionne le noyau Poetry."
        ) from exc
    print(f"{package_name} : {installed_version}")

print("Étape 1 validée : Python et les paquets vérifiés sont disponibles.")

Python du notebook : /Users/surelmanda/3-Mlops-Databricks-Projects/BNPL-Credit-Risk/.venv/bin/python
Version Python : 3.11.9
bnpl-credit-risk : 0.1.0
fastapi : 0.139.0
uvicorn : 0.51.0
httpx : 0.28.1
pydantic : 2.13.4
Étape 1 validée : Python et les paquets vérifiés sont disponibles.


## 2. Imports

Exécute la cellule. Elle doit terminer sans erreur.

In [7]:
# Outils pour lancer et arrêter le serveur local.
import os
import secrets
import subprocess
from pathlib import Path
from tempfile import TemporaryDirectory, TemporaryFile
from time import monotonic, perf_counter, sleep

# Clients HTTP et outils de présentation des résultats.
import httpx
import numpy as np
import pandas as pd
from fastapi.testclient import TestClient
from IPython.display import display

# Composants déjà implémentés dans le package du projet.
from bnpl_credit_risk.api.app import create_app
from bnpl_credit_risk.api.schemas import ApplicationRequest
from bnpl_credit_risk.api.settings import APISettings
from bnpl_credit_risk.inference.batch import BatchPredictor
from bnpl_credit_risk.models.registry import resolve_model_dir
from bnpl_credit_risk.settings import Settings, load_config

## 3. Vérifier le modèle

Attendu : une version publiée, `approved_for_inference=True` et `application_risk`.
En cas d’échec, termine la publication du notebook 05 avant de continuer.

In [8]:
import json

config = load_config()
settings = Settings()
models_dir = settings.resolve(settings.artifacts_dir) / "models"
model_dir = resolve_model_dir(models_dir, config.inference.model_version)
version = model_dir.name

# Vérifier les fichiers nécessaires au chargement d’ArtifactBundle.
required_artifacts = [
    "pipeline.joblib", "metadata.json", "metrics.json", "threshold.json", "feature_schema.json",
]
missing_artifacts = [name for name in required_artifacts if not (model_dir / name).is_file()]
assert not missing_artifacts, (
    f"Publication incomplète : {missing_artifacts}. Termine la publication du notebook 05."
)

metadata = json.loads((model_dir / "metadata.json").read_text(encoding="utf-8"))
assert metadata.get("approved_for_inference") is True, (
    "Ce modèle n’est pas approuvé. Termine l’évaluation et la publication du notebook 05."
)
assert metadata.get("risk_scope") == "application_risk", (
    "Cette API attend un modèle application_risk : les informations disponibles au checkout."
)

print(f"Dossier du modèle : {model_dir}")
print(f"Version sélectionnée : {version}")
print(f"Algorithme : {metadata.get('algorithm', 'non renseigné')}")
print(f"Approuvé pour l’inférence : {metadata['approved_for_inference']}")
print(f"Scope : {metadata['risk_scope']}")
print("Étape 3 validée : la publication du modèle est disponible pour la suite.")

Dossier du modèle : /Users/surelmanda/3-Mlops-Databricks-Projects/BNPL-Credit-Risk/artifacts/models/2026-08-28_131115
Version sélectionnée : 2026-08-28_131115
Algorithme : CatBoost
Approuvé pour l’inférence : True
Scope : application_risk
Étape 3 validée : la publication du modèle est disponible pour la suite.


## 4. Lancer l’API avec Poetry

La cellule lance `poetry run bnpl-api` en arrière-plan et crée automatiquement la clé d’accès.
Attends le message **API prête** avant de continuer.

Le port `8000` doit être libre : arrête tout serveur déjà lancé avec `Ctrl+C`, ou modifie `api_port`.

In [9]:
# Rejouer cette cellule ne doit pas démarrer une deuxième instance.
existing_process = globals().get("api_process")
if existing_process is not None and existing_process.poll() is None:
    print("L’API du notebook est déjà lancée :", globals()["live_url"] + "/health/ready")
else:
    config = load_config()
    settings = Settings()
    models_dir = settings.resolve(settings.artifacts_dir) / "models"
    version = resolve_model_dir(models_dir, config.inference.model_version).name
    api_port = 8000
    live_url = f"http://127.0.0.1:{api_port}"
    live_key = secrets.token_urlsafe(32)

    # Même clé et même version pour le serveur et notre requête de test.
    api_env = {
        **os.environ,
        "BNPL_API_API_KEY": live_key,
        "BNPL_API_ENVIRONMENT": "development",
        "BNPL_API_HOST": "127.0.0.1",
        "BNPL_API_PORT": str(api_port),
        "BNPL_API_MODEL_VERSION": version,
    }
    api_log = TemporaryFile(mode="w+t", encoding="utf-8")
    api_command = ["poetry", "run", "bnpl-api"]
    api_process = subprocess.Popen(
        api_command, cwd=settings.resolve("."), env=api_env,
        stdout=api_log, stderr=subprocess.STDOUT,
    )

    # Attendre le chargement du modèle ; une première connexion peut être refusée.
    try:
        deadline = monotonic() + 30
        with httpx.Client(base_url=live_url, timeout=2.0, trust_env=False) as http:
            while monotonic() < deadline:
                if api_process.poll() is not None:
                    api_log.seek(0)
                    raise RuntimeError(f"Échec du démarrage de l’API :\n{api_log.read()}")
                try:
                    ready = http.get("/health/ready")
                    # La clé aléatoire vérifie aussi que c’est bien notre serveur qui répond.
                    model_check = http.get("/v1/model", headers={"X-API-Key": live_key})
                    if ready.status_code == 200 and model_check.status_code == 200:
                        assert model_check.json()["model_version"] == version
                        break
                except httpx.TransportError:
                    pass
                sleep(0.2)
            else:
                raise TimeoutError("L’API n’est pas prête après 30 secondes.")
    except BaseException:
        # Nettoyer aussi en cas d’échec ou d’interruption manuelle de cette cellule.
        api_process.terminate()
        try:
            api_process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            api_process.kill()
            api_process.wait()
        api_log.close()
        raise
    print(f"API prête — version {version}")
    print(f"Vérifier l’API : {live_url}/health/ready")
    print(f"Documentation interactive : {live_url}/docs")

L’API du notebook est déjà lancée : http://127.0.0.1:8000/health/ready


## 5. Tester quatre requêtes HTTP

L’API doit être lancée. Attendu : **deux demandes valides → 200**, **deux invalides → 422**.
Les erreurs Pydantic sont détaillées localement ; l’API renvoie un message générique.

**Classe 0 : Repayment predicted. Classe 1 : Non-repayment predicted.**

In [ ]:
from pydantic import ValidationError

# 1. Demande valide.
live_payload_1 = {
    "user_id": 123,
    "age": 35,
    "employment_type": "Salaried",
    "monthly_income": 4000,
    "credit_score": 700,
    "purchase_amount": 300,
    "product_category": "Electronics",
    "bnpl_installments": 3,
    "app_usage_frequency": 10,
    "location": "USA",
    "transaction_date": "2026-01-01",
    "debt_to_income_ratio": 0.2,
}

# 2. Autre demande valide : des valeurs différentes, toujours conformes au contrat.
live_payload_2 = {
    **live_payload_1,
    "user_id": 124,
    "age": 28,
    "employment_type": "Self-Employed",
    "monthly_income": 2500,
    "credit_score": 620,
    "purchase_amount": 600,
    "bnpl_installments": 6,
    "location": "Canada",
    "debt_to_income_ratio": 0.4,
}

# 3. Erreur de type : une chaîne au lieu d’un nombre, refusée en mode strict.
live_payload_3 = {**live_payload_1, "user_id": 125, "monthly_income": "4000"}

# 4. Champ obligatoire manquant : age est supprimé de la demande.
live_payload_4 = {**live_payload_1, "user_id": 126}
live_payload_4.pop("age")

live_payloads = [
    ("1 — Demande valide", live_payload_1, 200),
    ("2 — Autre demande valide", live_payload_2, 200),
    ("3 — Revenu envoyé comme texte", live_payload_3, 422),
    ("4 — Âge manquant", live_payload_4, 422),
]
live_results = []
live_responses = []

print("=" * 72)
print("TEST DE 4 DEMANDES : 2 VALIDES ET 2 INVALIDES")
print("Pour chaque demande : validation Pydantic → appel API → résultat.")
print("=" * 72)

with httpx.Client(base_url=live_url, timeout=10.0, trust_env=False) as http:
    for test_name, live_payload, expected_status in live_payloads:
        print("\n" + "=" * 72)
        print(f"TEST {test_name}")
        print(f"Résultat attendu : HTTP {expected_status}")
        print("-" * 72)
        print("1. VALIDATION DES DONNÉES AVEC PYDANTIC")

        # Afficher précisément ce que Pydantic accepte ou rejette, sans changer l’API.
        try:
            ApplicationRequest.model_validate(live_payload)
            pydantic_status = "Accepté"
            print("Pydantic : données acceptées.")
        except ValidationError as exc:
            pydantic_status = "Rejeté"
            print("Pydantic : données rejetées. Voici le champ et la cause :")
            display(pd.DataFrame([
                {
                    "champ": ".".join(str(part) for part in error["loc"]),
                    "type_erreur": error["type"],
                    "message": error["msg"],
                }
                for error in exc.errors(include_input=False, include_context=False)
            ]))

        print("-" * 72)
        print("2. ENVOI DE LA DEMANDE À POST /v1/predict")
        # Envoyer aussi les cas invalides pour observer le vrai comportement HTTP.
        live_response = http.post(
            "/v1/predict", json=live_payload, headers={"X-API-Key": live_key},
        )
        response_body = live_response.json()
        print(f"HTTP obtenu : {live_response.status_code} — attendu : {expected_status}")
        assert live_response.status_code == expected_status, response_body

        print("-" * 72)
        print("3. LECTURE DE LA RÉPONSE")
        if expected_status == 200:
            print("Requête valide : voici la classe prédite par le modèle.")
            assert pydantic_status == "Accepté"
            assert response_body["user_id"] == live_payload["user_id"]
            assert response_body["model_version"] == version
            assert 0 <= response_body["default_probability"] <= 1
            assert response_body["default_risk_class"] == int(
                response_body["default_probability"] >= response_body["decision_threshold"]
            )
            assert response_body["request_id"] == live_response.headers["x-request-id"]
            # Libellé ajouté à l’affichage ; default_risk_class reste le code renvoyé par l’API.
            predicted_class = {0: "Repayment predicted", 1: "Non-repayment predicted"}[
                response_body["default_risk_class"]
            ]
            print(f"Classe prédite : {predicted_class} (code {response_body['default_risk_class']})")
            print(
                f"Probabilité estimée de non-remboursement : {response_body['default_probability']:.2%} — "
                f"seuil : {response_body['decision_threshold']:.2%}"
            )
            prediction_display = {"predicted_class": predicted_class, **response_body}
            display(pd.Series(prediction_display, name="Prédiction").to_frame())
        else:
            print("Demande rejetée : HTTP 422 est attendu, aucune prédiction n’est retournée.")
            print("L’API renvoie un message générique ; le détail Pydantic figure à l’étape 1.")
            # Un 422 est attendu : ne pas appeler raise_for_status(), qui arrêterait la boucle.
            assert pydantic_status == "Rejeté"
            assert response_body["error"]["code"] == "validation_error"
            assert response_body["error"]["request_id"] == live_response.headers["x-request-id"]
            display(pd.Series(response_body["error"], name="Erreur renvoyée par l’API").to_frame())

        print(f"TEST VALIDÉ : HTTP {live_response.status_code} correspond au résultat attendu.")
        print("=" * 72)
        live_responses.append(response_body)
        live_results.append({
            "test": test_name,
            "Pydantic": pydantic_status,
            "HTTP attendu": expected_status,
            "HTTP obtenu": live_response.status_code,
        })

print("\n" + "=" * 72)
print("BILAN DES 4 TESTS")
print("Les deux demandes valides sont acceptées et les deux invalides sont rejetées.")
print("=" * 72)
display(pd.DataFrame(live_results))

## 6. Préparer les tests complémentaires

Les cellules suivantes testent l’application en mémoire avec `TestClient`.
Exécute cette configuration avant de poursuivre.

In [11]:
config = load_config()
settings = Settings()
models_dir = settings.resolve(settings.artifacts_dir) / "models"
version = resolve_model_dir(models_dir, config.inference.model_version).name

# _env_file=None isole les tests des secrets enregistrés dans un .env local.
test_key = secrets.token_urlsafe(32)
api_settings = APISettings(
    api_key=test_key, environment="development", model_version=version,
    log_level="WARNING", _env_file=None,
)
headers = {"X-API-Key": test_key}
app = create_app(api_settings, settings)
print(f"Version à tester : {version}")

Version à tester : 2026-08-28_131115


## 7. Vérifier la santé et le modèle

Attendu : les assertions passent et le contrat du modèle s’affiche.

In [12]:
with TestClient(app) as http:
    assert http.get("/health/live").json() == {"status": "alive"}
    assert http.get("/health/ready").json() == {"status": "ready"}
    response = http.get("/v1/model", headers=headers)
    assert response.status_code == 200, response.text
    contract = response.json()
    assert contract["model_version"] == version
    assert contract["approved_for_inference"] is True
    assert contract["risk_scope"] == "application_risk"

assert app.state.service is None  # Le shutdown a bien libéré le service.
display(pd.Series(contract, name="Contrat API").to_frame())

,Contrat API
model_version,2026-08-28_131115
algorithm,CatBoost
risk_scope,application_risk
approved_for_inference,True
decision_threshold,0.293952
input_features,"[age, monthly_income, credit_score, purchase_a..."
required_request_fields,"[user_id, age, employment_type, monthly_income..."
bounds,"{'age': {'min': 18.0, 'max': 100.0}, 'monthly_..."
allowed_values,"{'employment_type': ['Salaried', 'Student', 'S..."


## 8. Vérifier une prédiction en mémoire

Attendu : une probabilité entre 0 et 1 et une classe cohérente avec le seuil.

In [ ]:
application = ApplicationRequest(
    user_id=123, age=35, employment_type="Salaried", monthly_income=4000,
    credit_score=700, purchase_amount=300, product_category="Electronics",
    bnpl_installments=3, app_usage_frequency=10, location="USA",
    transaction_date="2026-01-01", debt_to_income_ratio=0.2,
)
payload = application.model_dump(mode="json")
with TestClient(app) as http:
    response = http.post("/v1/predict", json=payload, headers=headers)
    assert response.status_code == 200, response.text
    prediction = response.json()
    assert prediction["request_id"] == response.headers["x-request-id"]
    assert 0 <= prediction["default_probability"] <= 1
    assert prediction["default_risk_class"] == int(
        prediction["default_probability"] >= contract["decision_threshold"]
    )
    assert prediction["scoring_timestamp"].endswith("Z")
    assert response.headers["cache-control"] == "no-store"

# Le libellé facilite la lecture du code numérique de la classe.
prediction_display = {
    "predicted_class": {0: "Repayment predicted", 1: "Non-repayment predicted"}[prediction["default_risk_class"]],
    **prediction,
}
display(pd.Series(prediction_display, name="Prédiction synthétique").to_frame())

## 9. Tester les erreurs

Ces requêtes sont volontairement invalides. Attendu : les statuts obtenus correspondent aux statuts attendus.

In [14]:
cases = [
    ("clé absente", payload, {}, 401),
    ("clé incorrecte", payload, {"X-API-Key": "invalid"}, 401),
    ("âge hors bornes", {**payload, "age": 12}, headers, 422),
    ("chaîne au lieu d'un nombre", {**payload, "monthly_income": "4000"}, headers, 422),
    ("nombre d'échéances invalide", {**payload, "bnpl_installments": 0}, headers, 422),
    ("date invalide", {**payload, "transaction_date": "2026-02-30"}, headers, 422),
    ("catégorie inconnue", {**payload, "employment_type": "Unknown"}, headers, 422),
    ("fuite de cible", {**payload, "default_flag": 1}, headers, 422),
    ("variable post-octroi", {**payload, "missed_payments": 1}, headers, 422),
    ("champ absent", {k: v for k, v in payload.items() if k != "age"}, headers, 422),
]
checks = []
with TestClient(app) as http:
    for name, body, auth, expected in cases:
        result = http.post("/v1/predict", json=body, headers=auth)
        assert result.status_code == expected, (name, result.text)
        assert result.json()["error"]["request_id"] == result.headers["x-request-id"]
        checks.append({"test": name, "attendu": expected, "obtenu": result.status_code})
    oversized = http.post(
        "/v1/predict", content="x" * (api_settings.max_body_bytes + 1),
        headers={**headers, "Content-Type": "application/json"},
    )
    assert oversized.status_code == 413
    checks.append({"test": "corps trop volumineux", "attendu": 413, "obtenu": 413})

display(pd.DataFrame(checks))

{"text": "2026-09-16 07:58:43.699 | WARNING  | bnpl_credit_risk.data.validation:validate:102 - Row-level validation issues: 1 / 1 rows flagged — {'out_of_bounds:age': 1}\n", "record": {"elapsed": {"repr": "0:01:10.620451", "seconds": 70.620451}, "exception": null, "extra": {"pipeline": "data.validation", "request_id": "f99683a3c82e494db0a7c62826259ef9"}, "file": {"name": "validation.py", "path": "/Users/surelmanda/3-Mlops-Databricks-Projects/BNPL-Credit-Risk/src/bnpl_credit_risk/data/validation.py"}, "function": "validate", "level": {"icon": "⚠️", "name": "WARNING", "no": 30}, "line": 102, "message": "Row-level validation issues: 1 / 1 rows flagged — {'out_of_bounds:age': 1}", "module": "validation", "name": "bnpl_credit_risk.data.validation", "process": {"id": 27293, "name": "MainProcess"}, "thread": {"id": 6332903424, "name": "AnyIO worker thread"}, "time": {"repr": "2026-09-16 07:58:43.699230+02:00", "timestamp": 1789538323.69923}}}
{"text": "2026-09-16 07:58:43.707 | WARNING  | bnp

,test,attendu,obtenu
0,clé absente,401,401
1,clé incorrecte,401,401
2,âge hors bornes,422,422
3,chaîne au lieu d'un nombre,422,422
4,nombre d'échéances invalide,422,422
5,date invalide,422,422
6,catégorie inconnue,422,422
7,fuite de cible,422,422
8,variable post-octroi,422,422
9,champ absent,422,422


## 10. Comparer l’API au batch

Prérequis : `data/processed/test_features.csv` disponible.
Attendu : **Parité API / batch validée sur 25 demandes**.

In [15]:
test_frame = pd.read_csv(settings.resolve(config.inference.input_path), nrows=25)
request_fields = list(ApplicationRequest.model_fields)
requests_frame = test_frame.loc[:, request_fields].copy()
records = requests_frame.to_dict(orient="records")

with TestClient(app) as http:
    realtime_rows = []
    for record in records:
        result = http.post("/v1/predict", json=record, headers=headers)
        assert result.status_code == 200, result.text
        realtime_rows.append(result.json())
realtime = pd.DataFrame(realtime_rows).set_index(config.data.id_column)

with TemporaryDirectory(prefix="bnpl-notebook07-") as directory:
    source = Path(directory) / "applications.csv"
    destination = Path(directory) / "predictions.csv"
    requests_frame.to_csv(source, index=False)
    BatchPredictor(config, settings).run(source, destination, model_version=version)
    batch = pd.read_csv(destination).set_index(config.data.id_column)

# Le batch conserve son nom historique ; aligner les colonnes pour la comparaison.
batch = batch.rename(columns={"predicted_default": "default_risk_class"})
batch = batch.loc[realtime.index]
np.testing.assert_allclose(
    realtime["default_probability"], batch["default_probability"], rtol=1e-10, atol=1e-12,
)
for column in ["default_risk_class", "risk_band", "model_version"]:
    assert realtime[column].equals(batch[column]), column
np.testing.assert_allclose(realtime["decision_threshold"], batch["decision_threshold"])
print(f"Parité API / batch validée sur {len(realtime)} demandes, version {version}.")

Parité API / batch validée sur 25 demandes, version 2026-08-28_131115.


## 11. Mesurer la latence

Exécute les 30 requêtes de test. Attendu : p50, p95, p99 et confirmation des métriques.
Ces mesures sont locales, en mémoire et sans réseau.

In [16]:
latencies_ms = []
with TestClient(app) as http:
    for _ in range(30):
        started = perf_counter()
        result = http.post("/v1/predict", json=payload, headers=headers)
        assert result.status_code == 200, result.text
        latencies_ms.append((perf_counter() - started) * 1000)
    metrics = http.get("/metrics", headers=headers)
    assert metrics.status_code == 200
    assert "bnpl_http_requests_total" in metrics.text
    assert "bnpl_http_request_duration_seconds" in metrics.text
    assert test_key not in metrics.text

latency_summary = pd.Series({
    "nombre de requêtes": len(latencies_ms),
    "p50 (ms)": np.percentile(latencies_ms, 50),
    "p95 (ms)": np.percentile(latencies_ms, 95),
    "p99 (ms)": np.percentile(latencies_ms, 99),
}, name="Latence locale indicative")
display(latency_summary.to_frame())
print("Métriques protégées et disponibles pour Prometheus.")

,Latence locale indicative
nombre de requêtes,30.000000
p50 (ms),8.689042
p95 (ms),10.238041
p99 (ms),11.465884


Métriques protégées et disponibles pour Prometheus.


## 12. Arrêter l’API

Exécute cette cellule à la fin des essais, même si un test a échoué.
Elle arrête uniquement le serveur créé par ce notebook.

In [17]:
# Arrêt gracieux, puis arrêt forcé seulement si le délai est dépassé.
if "api_process" in globals() and api_process.poll() is None:
    api_process.terminate()
    try:
        api_process.wait(timeout=35)
    except subprocess.TimeoutExpired:
        api_process.kill()
        api_process.wait()
if "api_log" in globals() and not api_log.closed:
    api_log.close()
print("L’API lancée par ce notebook est arrêtée.")

L’API lancée par ce notebook est arrêtée.
